### 패키지 설치 확인

In [1]:
import shutil
import sys

print("Python:", sys.version.split()[0])
print("가상환경 경로:", sys.prefix)
print("가상환경 안에서 실행 중?", ".venv" in sys.prefix)
print()

# OS 패키지 확인 (pip로 설치되지 않는 것들)
print("Poppler  :", "OK" if shutil.which("pdftoppm") else "✗ 미설치 — Path 확인 후 재시작")
print("Tesseract:", "OK" if shutil.which("tesseract") else "✗ 미설치 — Path 확인 후 재시작")
print()

# 파이썬 패키지 확인
for mod in ["langchain", "unstructured", "chromadb", "tiktoken", "PIL"]:
    try:
        __import__(mod)
        print(f"{mod:14s}: OK")
    except ImportError:
        print(f"{mod:14s}: ✗ 미설치")

Python: 3.11.9
가상환경 경로: c:\workspace\multimodal-rag\.venv
가상환경 안에서 실행 중? True

Poppler  : OK
Tesseract: OK

langchain     : OK
unstructured  : OK
chromadb      : OK
tiktoken      : OK
PIL           : OK


### 추출 함수

In [2]:
from langchain_text_splitters import CharacterTextSplitter
from unstructured.partition.pdf import partition_pdf

def extract_pdf_elements(path, fname):
    """
    PDF: 파일에서 이미지, 테이블, 텍스트 블록을 추출합니다.
    path: 파일 경로(추출된 이미지 파일이 저장될 위치)
    fname: 파일 이름
    """
    return partition_pdf(
        filename=path+fname,
        # 레이아웃 분석 모델로 정밀 추출 (교재에 없음! 필수 추가)
        strategy="hi_res",
        # PDF에서 이미지를 추출
        extract_images_in_pdf=True,
        # 테이블 구조를 추론
        infer_table_structure=True,
        # 타이틀을 기준으로 텍스트를 블록으로 분할
        chunking_strategy="by_title",
        # 최대 4000자로 텍스트 블록을 제한
        max_characters=4000,
        # 3800자 이후에 새로운 블록 생성
        new_after_n_chars=3800,
        # 2000자 이하의 텍스트는 결합
        combine_text_under_n_char=2000,
        # 이미지가 저장될 경로 설정
        # image_output_dir_path=path,
        extract_image_block_output_dir=os.path.join(path, "figures"),
    )

### 추출한 요소를 종류별로 분류

In [3]:
def categorize_elements(raw_pdf_elements):
    """
    PDF에서 추출한 요소들을 테이블과 텍스트로 분류합니다.
    raw_pdf_elements: unstructured.documents.elements 리스트
    """
    tables = []
    texts = []
    for element in raw_pdf_elements:
        if "unstructured.documents.elements.Table" in str(type(element)):
            tables.append(str(element))
        elif "unstructured.documents.elements.CompositeElement" in str(type(element)):
            texts.append(str(element))
    return tables, texts

### 경로 설정 및 실행

In [4]:
import os

# 현재 작업 디렉터리 경로를 반환
current_directory = os.getcwd()

# fame: 처리할 PDF 파일 이름 / fpath: PDF가 저장된 폴더 경로
fname = "invest.pdf"
fpath = os.path.join(os.path.dirname(current_directory), "data/")

print("현재 스크립트의 위치:", current_directory)
print("pdf 위치:", fpath)

#실행하면 2~3분 소요, 멈춘 것이 아니니 기다리세요.
raw_pdf_elements = extract_pdf_elements(fpath, fname)
tables, texts = categorize_elements(raw_pdf_elements)

Thank you for using Unstructured Open Source! Try Unstructured's commercial offering for improved output quality. 10,000 free pages to start. Learn more: https://transform.unstructured.io/?utm_source=unstructured_oss Hide this notice: UNSTRUCTURED_DISABLE_NOTICE=1


현재 스크립트의 위치: c:\workspace\multimodal-rag\notebooks
pdf 위치: c:\workspace\multimodal-rag\data/


No languages specified, defaulting to English.
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 2741.97it/s]


### 텍스트 분할

In [5]:
# chunk_size=2000: 각 텍스트 블록의 최대 크기
# chunk_overlap=200: 청크 간 200자의 겹침을 허용해 중요한 정보를 잃지 않도록 함
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 2000,
    chunk_overlap = 200,
)

# 텍스트 리스트를 하나의 긴 문자열로 결합한 뒤 2000자 단위로 분할
joined_texts = " ".join(texts)
texts_2k_token = text_splitter.split_text(joined_texts)

print("분할 후 블록 수:", len(texts_2k_token))
print("원본 블록 수:", len(texts))
print(texts[0])
print(type(texts_2k_token))

분할 후 블록 수: 6
원본 블록 수: 9
ISSN 2288-7083 경제전망 Indigo Book 2026년 2월 성장 2%대 반등, 부문별 온도차

ISSN 2288-7083

경제전망

er Oy BANK OF KOREA

부문별 담당자

부 문

담 당 팀

담 당 자
<class 'list'>


### API 키 불러오기

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OpenAI API 키가 없습니다. 한번 더 확인 부탁드립니다.")

### 요약 체인 구성

In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_ollama.llms import OllamaLLM

def generate_text_summaries(texts, tables, summarize_texts=False):
    """
    텍스트 및 표 데이터를 요약하며 검색에 활용할 수 있는 요약본을 생성합니다.
    texts: 텍스트 리스트
    tables: 표 리스트
    summarize_texts: 텍스트 요약 여부 (기본값: False)
    """

    # prompt 한국어 버전
    prompt_text_kor ="""
    당신은 표와 텍스트를 요약하며 검색에 활용할 수 있도록 돕는 도우미 입니다.
    이 요약본들은 임베딩되어 원본 텍스트나 표 요소를 검색하는 데 사용될 것 입니다.
    주어진 표나 텍스트의 내용을 검색에 최적화된 간결한 요약으로 작성해주세요.
    요약할 표 또는 텍스트: {element}
    """

    prompt = ChatPromptTemplate.from_template(template=prompt_text_kor)
    
    # 모델: GPT-4o-mini or Llama3.1 모델 사용
    # llamaModel = OllamaLLM(model="llama3.1:8b")
    model = ChatOpenAI(temperature=0, model="gpt-4o-mini")

    # 요약 체인 = 프롬프트 + LLM + 출력 파서
    summarize_chain = {"element": lambda x:x} | prompt | model | StrOutputParser()
    text_summaries = []
    table_summaries = []

    # 텍스트 요약을 활성화한 경우
    if texts and summarize_texts:
        text_summaries = summarize_chain.batch(texts, {"max_concurrency":5})
    #텍스트 요약 비활성화
    elif texts:
        text_summaries = texts

    # 테이블 데이터 요약
    if tables:
        table_summaries = summarize_chain.batch(tables, {"max_concurrency":5})
    return text_summaries, table_summaries

text_summaries, table_summaries = generate_text_summaries(
    texts_2k_token, tables, summarize_texts=True
)
print(text_summaries)
print(table_summaries)

"""
max_concurrency=5는 한번에 5개씩 요약하라는 뜻, 숫자를 올리면 더 빨라지지만 API 요청 제한 위험
"""

['**경제전망 요약 (2026년 2월)**\n\n- **성장률**: 2026년 한국 경제는 2.0% 성장할 것으로 전망됨. 반도체 경기 개선과 양호한 세계경제 흐름이 긍정적 요인으로 작용.\n- **물가**: 소비자물가는 2.2% 상승할 것으로 예상되며, 이는 일부 품목의 비용 상승 압력에 기인.\n- **세계경제**: AI 투자와 완화적 통화·재정정책 덕분에 세계경제는 3% 초반의 성장세를 유지할 전망. 그러나 미국의 관세 관련 불확실성이 증가.\n- **국가별 전망**:\n  - **미국**: AI 투자 확대와 정책 지원으로 견조한 성장 예상.\n  - **유로지역**: 완화적 금융 여건과 재정 지출 확대에 힘입어 완만한 성장 지속.\n  - **중국**: 부동산 침체와 내수 부진으로 성장세 둔화, 그러나 내수 부양 정책으로 완충 가능성.\n- **세계교역**: 무역환경의 불확실성 속에서도 AI 관련 교역 호조와 EU-인도 FTA 체결로 다소 완화될 전망. \n\n이 요약은 경제전망의 주요 내용을 간결하게 정리하여 검색에 최적화된 형태로 제공됩니다.', '**요약:**\n\n- **국제유가**: 브렌트유는 미-이란 군사적 긴장으로 상승했으나, 연중 초과공급으로 완만한 하락세 예상.\n  \n- **대내여건**: IT기업 실적 호조로 소득여건 개선, 경제심리 양호. 비IT 부문 부진과 외환·금융시장 변동성이 리스크 요인.\n\n- **임금상승률**: IT기업 실적 호조로 완만한 개선 전망.\n\n- **시장금리**: 기준금리 인하 기대 약화로 회사채 금리 상승. 원/달러 환율 변동성 증가, 최근 1,400원대 중반으로 하락.\n\n- **주택가격**: 수도권 중심 상승세, 정부 대책으로 속도 둔화.\n\n- **관세 변화**: 미국, 한국에 대한 관세율 유지. 글로벌 평균관세율 소폭 하락. 펜타닐관세는 중국 등에서 하락.\n\n- **관세 전제**: 임시관세 만료 후 새로운 관세 부과 예상, 기존 관세율로 복귀 가능성.', '미국 정부는 관세 위법 판결에 

'\nmax_concurrency=5는 한번에 5개씩 요약하라는 뜻, 숫자를 올리면 더 빨라지지만 API 요청 제한 위험\n'

### Base64 인코딩

In [8]:
import base64
import os
import requests
import json
from langchain_core.messages import HumanMessage

def encode_image(image_path):
    """이미지를 base64 문자열로 변환"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

### OpenAI 모델로 이미지 요약

In [ ]:
def image_summarize(img_base64, prompt):
    """이미지 요약 생성"""
    # ❌ 교재: model="gpt-4-vision-preview" → 지원 종료(deprecated)
    # ✅ 현재: gpt-4o-mini 가 비전 기능을 포함하며 비용도 훨씬 저렴합니다
    chat = ChatOpenAI(model="gpt-4o-mini", max_tokens=1024)

    msg = chat.invoke(
        [HumanMessage(
            content=[
                {"type":"text", "text":prompt},
                {"type":"image_url", "image_url":{
                    "url":f"data:image/jpeg;base64,{img_base64}"
                }}
            ])]
    )
    return msg.content

### LLaVA(오픈소스) 모델로 이미지 요약

In [10]:
def image_summarize_llava(img_base64, prompt):
    """LLaVA를 사용하여 이미지 요약 생성"""
    payload = {
        "model": "llava:7b",
        "prompt": prompt,
        "images": [img_base64],
    }

    response = requests.post("http://localhost:11434/api/generate", json=payload)
    print("Status Code: ", response.status_code)

    if response.status_code == 200:
        try:
            # 응답을 줄 단위로 처리
            full_response = ""
            for line in response.iter_lines():
                if line:
                    json_line = json.loads(line)
                    if "response" in json_line:
                        full_response += json_line["response"]
            print(full_response)
            return full_response
        except json.JSONDecodeError as e:
            return f"JSON 파싱 오류: {str(e)}\n응답 내용: {response.text}"
    else:
        return f"Error: {response.status_code}, {response.text}"

### 폴더 내 전체 이미지 요약

In [12]:
def generate_img_summaries(path):
    """
    이미지의 요약과 Base64 인코딩 문자열을 생성
    path: Unstuctured가 추출한 .jpg의 경로
    """
    img_base64_list = []
    image_summaries = []

    # Prompt_kor 한국어
    prompt_kor ="""You are an assistant tasked with summarizing images for retrieval.
    These summaries will be embedded and used to retrieve the raw image.
    Provide a concise summary of the image that is well optimized for retrieval.
    The summary should be written in Korean (Hangul)."""

    # Prompt 영어
    prompt = """You are an assistant tasked with summarizing images for retrieval.
    These summaries will be embedded and used to retrieve the raw image.
    Provide a concise summary of the image that is well optimized for retrieval."""

    # 주어진 경로에서 파일 목록을 가져와 정렬, 각 파일을 처리
    for img_file in sorted(os.listdir(path)):
        # 파일이 .jpg, .png, .jpeg 확장자 중 하나일 경우만 처리
        if img_file.endswith((".jpg", ".png", ".jpeg")):
            img_path = os.path.join(path, img_file)
            base64_image = encode_image(img_path)
            img_base64_list.append(base64_image)
            image_summaries.append(image_summarize(base64_image, prompt_kor))

    return img_base64_list, image_summaries


img_base64_list, image_summaries = generate_img_summaries(fpath + "figures/")
print(image_summaries[0])

C:\Users\user\AppData\Local\Temp\ipykernel_12700\4021852751.py:32: UserWarning: WARNING! max_token is not default parameter.
                max_token was transferred to model_kwargs.
                Please confirm that max_token is what you intended.
  img_base64_list, image_summaries = generate_img_summaries(fpath + "figures/")


TypeError: Completions.create() got an unexpected keyword argument 'max_token'